# 6.29 — Capsule Networks

Capsule networks replace scalar “is this feature present?” activations with small vectors whose **length** means presence and whose **direction** carries pose-like information. The central mechanism is routing-by-agreement: lower capsules make prediction vectors for possible upper capsules, softmax couplings decide where each lower capsule sends its evidence, and agreement between a vote and an upper capsule strengthens that route.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build capsule networks one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so routing agreement is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + linear algebra for capsule vectors and routing.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for tiny demonstrations.

### 1. Capsules store activation as vector length

A standard neuron returns one scalar. A capsule returns a vector: the **length** says how strongly an entity exists, while the **direction** can encode pose, thickness, orientation, or another structured property. This is why capsules try to preserve part information instead of throwing it away with pooling.

In [ ]:
caps_w = np.array([[0.9, 0.2], [0.1, 0.8], [0.3, 0.4]])  # three 2-D lower capsule outputs.
lengths_w = np.linalg.norm(caps_w, axis=1)  # activation strength is vector length.
print("capsule vectors:\n", caps_w)  # inspect pose-carrying directions.
print("activation lengths:", np.round(lengths_w, 3))  # inspect existence strengths.
assert np.allclose(np.round(lengths_w, 3), [0.922, 0.806, 0.5])  # concrete length checks.

▶ What you'll see: three short vectors with different directions and activation lengths.

In [ ]:
plt.figure(figsize=(4, 3.4))  # create a compact pose-space plot.
for i_w, v_w in enumerate(caps_w):  # draw each capsule vector as an arrow.
    plt.arrow(0, 0, v_w[0], v_w[1], head_width=0.04, length_includes_head=True)
    plt.text(v_w[0] + 0.03, v_w[1], f"cap {i_w}, |v|={lengths_w[i_w]:.2f}")
plt.xlim(0, 1.1); plt.ylim(0, 1.0); plt.xlabel("pose axis 0"); plt.ylabel("pose axis 1")
plt.title("1: activation is length, pose is direction"); plt.show()

▶ What you'll see: arrows with visible directions; the longer arrows are more active capsules.

*Why it's done this way: a scalar activation can say “feature present,” but a vector can say “feature present with this pose.” Length and direction split confidence from geometry, which gives routing more information than max pooling can keep.*

### 2. A tiny affine signal still shapes raw evidence

The lesson’s scratch pass is an ordinary affine map followed by a gate. With inputs $x=[1.5,-0.5]$, weights $[1.8,-0.4]$, and bias $0.8$, the signal is $3.7$ before and after ReLU because it is already positive.

In [ ]:
x_w = np.array([1.5, -0.5])  # two visible inputs from the lesson arithmetic.
w_w = np.array([1.8, -0.4])  # weights chosen so the computation is inspectable.
b0_w = 0.8  # bias term from the lesson block.
affine_w = float(w_w @ x_w + b0_w)  # 1.8*1.5 + -0.4*(-0.5) + 0.8.
gated_w = max(0.0, affine_w)  # ReLU gate keeps positive signals and clips negative ones.
print("affine:", round(affine_w, 3), "gated:", round(gated_w, 3))
assert round(affine_w, 3) == 3.7 and round(gated_w, 3) == 3.7

▶ What you'll see: the raw affine score is `3.7`, and the gate keeps it unchanged.

In [ ]:
parts_w = np.array([w_w[0] * x_w[0], w_w[1] * x_w[1], b0_w])  # decompose the affine score.
plt.figure(figsize=(4.4, 3))
plt.bar(["1.8·1.5", "-0.4·-0.5", "bias"], parts_w, color=["teal", "teal", "gray"])
plt.axhline(0, color="black", linewidth=0.8); plt.title("2: affine pieces sum to 3.7")
plt.ylabel("contribution"); plt.xticks(rotation=15); plt.show()

▶ What you'll see: two input contributions plus the bias add to the signal used downstream.

*Why it's done this way: capsules are special at the routing stage, but they still sit inside differentiable networks. The affine-and-gate pass shows how local numerical scale is created before routing decisions amplify or suppress evidence.*

### 3. Lower capsules make prediction vectors for possible wholes

Each lower capsule predicts what every upper capsule should look like if that part belongs to that whole. If lower capsule $i$ has output $u_i$, then a learned matrix $W_{ij}$ produces a vote $\hat u_{j|i}=W_{ij}u_i$. These votes are the raw “I think the whole is here with this pose” messages.

In [ ]:
u_w = np.array([[0.8, 0.1], [0.2, 0.7], [0.6, 0.3]])  # three lower capsules.
W_w = np.array([[[1.0, 0.0], [0.0, 0.8]], [[0.4, 0.6], [0.7, 0.2]],
                [[0.2, 0.9], [0.8, 0.1]], [[0.9, 0.1], [0.1, 0.9]],
                [[0.6, 0.3], [0.2, 0.7]], [[0.3, 0.7], [0.9, 0.1]]]).reshape(3, 2, 2, 2)  # W[i,j].
votes_w = np.einsum("ijab,ib->ija", W_w, u_w)  # votes[i,j] = W[i,j] @ u[i].
print("votes shape:", votes_w.shape)  # lower capsules x upper capsules x pose dims.
print("first lower capsule votes:\n", np.round(votes_w[0], 3))
assert votes_w.shape == (3, 2, 2)

▶ What you'll see: each of 3 lower capsules casts one 2-D prediction vector for each of 2 upper capsules.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
for j_w, color_w in enumerate(["seagreen", "darkorange"]):  # plot votes aimed at each possible upper capsule.
    plt.scatter(votes_w[:, j_w, 0], votes_w[:, j_w, 1], s=80, color=color_w, label=f"votes to upper {j_w}")
plt.xlabel("pose axis 0"); plt.ylabel("pose axis 1"); plt.legend()
plt.title("3: prediction vectors for two possible wholes"); plt.show()

▶ What you'll see: votes aimed at the same upper capsule form point clouds in pose space.

*Why it's done this way: routing cannot ask whether parts agree unless every part first predicts a common upper-capsule pose. The matrices $W_{ij}$ translate each part’s local coordinate system into each candidate whole’s coordinate system.*

### 4. Softmax couplings make each lower capsule choose where to send evidence

Routing logits $b_{ij}$ are turned into couplings $c_{ij}=\mathrm{softmax}_j(b_{ij})$. The softmax is over upper capsules for each fixed lower capsule, so every row sums to 1: a lower capsule distributes its evidence across possible parents rather than creating or destroying total responsibility.

In [ ]:
b_w = np.array([[3.7, 0.4], [0.2, 1.1], [0.6, 0.3]])  # routing logits for 3 lower x 2 upper capsules.
exp_w = np.exp(b_w - np.max(b_w, axis=1, keepdims=True))  # stable exponentials row by row.
c_w = exp_w / exp_w.sum(axis=1, keepdims=True)  # softmax over upper capsules.
print("couplings:\n", np.round(c_w, 3))
print("row sums:", np.round(c_w.sum(axis=1), 3))
assert round(float(c_w[0, 0]), 3) == 0.964  # lesson score 3.7 against baseline 0.4.

▶ What you'll see: the first lower capsule sends about 96.4% of its evidence to upper capsule 0.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(c_w, cmap="viridis", aspect="auto")
plt.colorbar(label="coupling c_ij"); plt.xlabel("upper capsule j"); plt.ylabel("lower capsule i")
plt.title("4: softmax routing couplings"); plt.show()

▶ What you'll see: a heatmap where each row is a probability distribution over possible parents.

*Why it's done this way: exponentials convert relative logits into positive weights, and row normalization turns those weights into a conserved routing budget. A big logit matters only compared with alternatives, which is exactly what a routing decision needs.*

### 5. Weighted sums collect routed votes into upper pre-activations

Once couplings are known, upper capsule $j$ receives $s_j=\sum_i c_{ij}\hat u_{j|i}$. This is a weighted average-like sum of votes: strongly coupled lower capsules influence the upper pose most, while weakly coupled capsules barely move it.

In [ ]:
s_w = np.einsum("ij,ija->ja", c_w, votes_w)  # weighted sum of votes into each upper capsule.
print("upper pre-activations s_j:\n", np.round(s_w, 3))
print("lengths before squash:", np.round(np.linalg.norm(s_w, axis=1), 3))
assert s_w.shape == (2, 2)

▶ What you'll see: two upper pre-activation vectors, one per candidate whole.

In [ ]:
plt.figure(figsize=(4.2, 3.4))
for j_w, color_w in enumerate(["seagreen", "darkorange"]):
    plt.arrow(0, 0, s_w[j_w, 0], s_w[j_w, 1], head_width=0.035, color=color_w, length_includes_head=True)
    plt.text(s_w[j_w, 0], s_w[j_w, 1], f"s{j_w}")
plt.xlim(0, 1.6); plt.ylim(0, 1.2); plt.xlabel("pose axis 0"); plt.ylabel("pose axis 1")
plt.title("5: routed votes become upper vectors"); plt.show()

▶ What you'll see: the upper pre-activation vectors point where their routed votes collectively agree.

*Why it's done this way: summing votes after weighting is the mathematical bridge from local part predictions to a whole-object hypothesis. If many parts route to the same upper capsule and point similarly, the resulting vector grows and stabilizes.*

### 6. Squashing keeps vector length below one without losing direction

Capsule networks use a squash nonlinearity such as $v_j=\frac{\|s_j\|^2}{1+\|s_j\|^2}\frac{s_j}{\|s_j\|}$. Small vectors shrink toward 0, large vectors approach length 1, and direction is preserved. That makes length interpretable as probability-like existence while retaining pose direction.

In [ ]:
def squash_w(s_w):  # capsule squash nonlinearity for a batch of vectors.
    norm_w = np.linalg.norm(s_w, axis=-1, keepdims=True)  # vector lengths.
    scale_w = (norm_w ** 2) / (1.0 + norm_w ** 2)  # saturating length factor.
    return scale_w * s_w / (norm_w + 1e-8)  # preserve direction and guard zero length.

v_w = squash_w(s_w)  # convert upper pre-activations into capsule outputs.
print("squashed outputs v_j:\n", np.round(v_w, 3))
print("lengths after squash:", np.round(np.linalg.norm(v_w, axis=1), 3))
assert np.all(np.linalg.norm(v_w, axis=1) < 1.0)

▶ What you'll see: upper vectors keep their direction but have lengths safely below 1.

In [ ]:
raw_norms_w = np.linspace(0, 4, 100)  # possible input lengths.
squashed_norms_w = raw_norms_w ** 2 / (1 + raw_norms_w ** 2)  # output lengths from the squash formula.
plt.figure(figsize=(4.4, 3))
plt.plot(raw_norms_w, squashed_norms_w, color="purple")
plt.xlabel("input length ||s||"); plt.ylabel("output length ||v||")
plt.title("6: squash saturates length below 1"); plt.show()

▶ What you'll see: the curve starts near 0 and approaches 1 without crossing it.

*Why it's done this way: routing needs vector length to behave like confidence. Linear length would grow without bound, while a hard threshold would kill gradients; the squash formula gives smooth saturation and preserves pose direction.*

### 7. Agreement updates routing logits

Dynamic routing repeats a simple loop: compute couplings, collect votes, squash, then add agreement $\hat u_{j|i}\cdot v_j$ to $b_{ij}$. If a lower capsule’s vote points in the same direction as an upper capsule output, its logit for that upper capsule increases next round.

In [ ]:
b_route_w = np.zeros((3, 2))  # start with no routing preference.
history_w = []  # record couplings over routing iterations.
for it_w in range(3):  # run three dynamic-routing iterations.
    e_w = np.exp(b_route_w - np.max(b_route_w, axis=1, keepdims=True))  # stable softmax numerators.
    c_route_w = e_w / e_w.sum(axis=1, keepdims=True)  # row-wise routing couplings.
    s_route_w = np.einsum("ij,ija->ja", c_route_w, votes_w)  # upper pre-activations.
    v_route_w = squash_w(s_route_w)  # upper capsule outputs.
    agreement_w = np.einsum("ija,ja->ij", votes_w, v_route_w)  # vote-output dot products.
    b_route_w = b_route_w + agreement_w  # reinforce agreeing routes.
    history_w.append(c_route_w.copy())  # store couplings before the update.
print("final couplings:\n", np.round(history_w[-1], 3))
assert np.allclose(history_w[0].sum(axis=1), 1.0)

▶ What you'll see: couplings begin uniform and become more decisive after agreement updates.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot([h_w[0, 0] for h_w in history_w], marker="o", label="lower 0 → upper 0")
plt.plot([h_w[0, 1] for h_w in history_w], marker="s", label="lower 0 → upper 1")
plt.ylim(0, 1); plt.xlabel("routing iteration"); plt.ylabel("coupling")
plt.title("7: agreement shifts routing mass"); plt.legend(); plt.show()

▶ What you'll see: one route gains probability while the competing route loses it.

*Why it's done this way: agreement is a dot product because direction alignment is the pose-consistency test. Adding agreement to logits makes routing iterative: current whole hypotheses decide which part votes should count more in the next pass.*

### 8. Margin loss and small updates train capsule outputs

Capsules are often trained with a margin loss on output lengths: present classes should have length above $m^+$, absent classes below $m^-$. The lesson’s scalar update $2.000-0.090\cdot1.350=1.879$ is the same training idea in miniature: small reliable steps change parameters without wrecking the routing geometry.

In [ ]:
length_cls_w = np.array([0.82, 0.18])  # two class capsule lengths.
target_w = np.array([1.0, 0.0])  # class 0 is present, class 1 is absent.
m_plus_w, m_minus_w, lam_loss_w = 0.9, 0.1, 0.5  # capsule margin-loss constants.
loss_terms_w = target_w * np.maximum(0, m_plus_w - length_cls_w) ** 2 + lam_loss_w * (1 - target_w) * np.maximum(0, length_cls_w - m_minus_w) ** 2
print("margin loss terms:", np.round(loss_terms_w, 4), "total:", round(float(loss_terms_w.sum()), 4))
assert round(float(loss_terms_w.sum()), 4) == 0.0096

▶ What you'll see: the present class is penalized for being below 0.9, and the absent class for being above 0.1.

In [ ]:
theta_w, eta_w, grad_w = 2.0, 0.09, 1.35  # lesson parameter, learning rate, and gradient.
theta_new_w = theta_w - eta_w * grad_w  # one gradient-descent step.
normalized_w = (3.7 - 1.0) / np.sqrt(0.250 + 0.00001)  # lesson scale bookkeeping.
mem_kb_w = 3 * 128 * 4 / 1024  # three 128-D float32 capsule vectors.
print("theta after update:", round(theta_new_w, 3))
print("normalized signal:", round(normalized_w, 3), "memory KB:", round(mem_kb_w, 3))
assert round(theta_new_w, 3) == 1.879 and round(normalized_w, 3) == 5.4 and round(mem_kb_w, 3) == 1.5

▶ What you'll see: the parameter moves a little, the score is 5.4 standard units above the mean, and the tiny capsule block costs 1.5 KB.

*Why it's done this way: the loss gives vector lengths a supervised meaning, the update changes parameters gradually, and the normalization/memory numbers remind us that routing is only useful if scales and hardware cost remain controlled.*

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, vector norms, softmaxes, and from-scratch routing loops.
import matplotlib.pyplot as plt  # Import Matplotlib for the arrows, heatmaps, and loss curves used to debug capsules.
np.random.seed(0)  # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Measure capsule activation length

**Goal.** Compute vector lengths, because capsule activation strength is the norm of a pose vector rather than a scalar neuron value. We build it in 2 steps.

In [ ]:
caps_b1 = np.array([[0.9, 0.2], [0.1, 0.8], [0.3, 0.4]])  # Store three 2-D capsule outputs.
lengths_b1 = np.linalg.norm(caps_b1, axis=1)  # Compute one activation length per capsule.
print("lengths:", np.round(lengths_b1, 3))  # Inspect confidence-like activation strengths.
assert np.allclose(np.round(lengths_b1, 3), np.array([0.922, 0.806, 0.5]))  # Verify concrete norm values.

▶ What you'll see: each capsule has one activation length while still keeping a 2-D direction.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact bar chart.
plt.bar(["cap0", "cap1", "cap2"], lengths_b1, color="teal")  # Visualize capsule existence strengths.
plt.title("Basic 1: capsule lengths")  # Title the plot.
plt.ylabel("||u_i||")  # Label the norm axis.
plt.show()  # Display the bar chart.

▶ What you'll see: longer bars correspond to more active capsules.

👀 Takeaway: capsule networks encode “how present” with vector length and leave direction for pose information.

### Basic 2 — Preserve pose direction separately from length

**Goal.** Normalize capsule vectors to unit directions, because routing compares pose directions while activation length carries confidence. We build it in 2 steps.

In [ ]:
caps_b2 = np.array([[0.9, 0.2], [0.1, 0.8], [0.3, 0.4]])  # Recreate three capsule vectors locally.
norms_b2 = np.linalg.norm(caps_b2, axis=1, keepdims=True)  # Compute lengths for normalization.
dirs_b2 = caps_b2 / norms_b2  # Divide each vector by its length to isolate direction.
print("unit directions:\n", np.round(dirs_b2, 3))  # Inspect pose direction without activation scale.
assert np.allclose(np.round(np.linalg.norm(dirs_b2, axis=1), 3), np.ones(3))  # Verify unit length.

▶ What you'll see: the vectors now have length 1 but still point in their original directions.

In [ ]:
plt.figure(figsize=(4, 3.4))  # Create a compact direction plot.
for i_b2, d_b2 in enumerate(dirs_b2):  # Draw unit pose directions.
    plt.arrow(0, 0, d_b2[0], d_b2[1], head_width=0.04, length_includes_head=True)
    plt.text(d_b2[0], d_b2[1], f"dir {i_b2}")
plt.xlim(0, 1.1); plt.ylim(0, 1.1); plt.title("Basic 2: pose directions")
plt.xlabel("axis 0"); plt.ylabel("axis 1"); plt.show()

▶ What you'll see: all arrows reach the unit circle but point in different directions.

👀 Takeaway: separating length from direction lets capsules represent confidence and geometry at the same time.

### Basic 3 — Reproduce the lesson affine-and-gate signal

**Goal.** Compute the lesson’s two-input scratch pass, because even capsule models rely on ordinary differentiable building blocks before routing. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1.5, -0.5])  # Define the two input values from the lesson block.
w_b3 = np.array([1.8, -0.4])  # Define the two weights from the worked arithmetic.
bias_b3 = 0.8  # Define the bias from the lesson block.
affine_b3 = float(w_b3 @ x_b3 + bias_b3)  # Compute 1.8*1.5 + -0.4*(-0.5) + 0.8.
print("affine signal:", round(affine_b3, 3))  # Inspect the raw signal.
assert round(affine_b3, 3) == 3.7  # Verify the lesson number.

▶ What you'll see: the affine signal equals 3.7.

In [ ]:
gated_b3 = max(0.0, affine_b3)  # Apply a ReLU gate to keep positive signals.
print("gated signal:", round(gated_b3, 3))  # Inspect the post-gate signal.
plt.figure(figsize=(4, 3))  # Create a before-after plot.
plt.bar(["affine", "gated"], [affine_b3, gated_b3], color=["gray", "seagreen"])  # Compare raw and gated signals.
plt.title("Basic 3: gate keeps positive evidence")  # Title the plot.
plt.ylabel("signal")  # Label the signal axis.
plt.show()  # Display the plot.

▶ What you'll see: both bars are equal because the signal is already positive.

👀 Takeaway: local feature signals must be shaped before they become capsule votes or routing logits.

### Basic 4 — Make one prediction vector

**Goal.** Transform a lower capsule into a vote for an upper capsule, because routing needs each part to predict a whole pose. We build it in 2 steps.

In [ ]:
u_b4 = np.array([0.8, 0.1])  # Define one lower capsule output.
W_b4 = np.array([[1.0, 0.0], [0.0, 0.8]])  # Define a small transformation matrix for one possible parent.
vote_b4 = W_b4 @ u_b4  # Compute the prediction vector u_hat = W u.
print("vote:", np.round(vote_b4, 3))  # Inspect the part's predicted whole pose.
assert np.allclose(vote_b4, np.array([0.8, 0.08]))  # Verify the matrix-vector product.

▶ What you'll see: the lower capsule becomes a parent-specific prediction vector.

In [ ]:
plt.figure(figsize=(4, 3.4))  # Create a vector comparison plot.
plt.arrow(0, 0, u_b4[0], u_b4[1], head_width=0.035, color="gray", length_includes_head=True, label="u")  # Draw input capsule.
plt.arrow(0, 0, vote_b4[0], vote_b4[1], head_width=0.035, color="teal", length_includes_head=True, label="W u")  # Draw vote.
plt.xlim(0, 1); plt.ylim(0, 0.25); plt.legend(); plt.title("Basic 4: transformed vote")
plt.xlabel("pose axis 0"); plt.ylabel("pose axis 1"); plt.show()

▶ What you'll see: the vote has the same first coordinate and a scaled second coordinate.

👀 Takeaway: learned transformations translate part pose into a candidate whole-pose coordinate system.

### Basic 5 — Softmax two routing logits

**Goal.** Convert two logits into routing probabilities, because each lower capsule must split its evidence across possible upper capsules. We build it in 2 steps.

In [ ]:
logits_b5 = np.array([3.7, 0.4])  # Compare the lesson score against the baseline.
exp_b5 = np.exp(logits_b5)  # Exponentiate both scores.
prob_b5 = exp_b5 / np.sum(exp_b5)  # Normalize into probabilities.
print("exp values:", np.round(exp_b5, 3))  # Inspect e^3.7 and e^0.4.
print("routing probs:", np.round(prob_b5, 3))  # Inspect the softmax result.
assert round(float(prob_b5[0]), 3) == 0.964  # Verify the lesson probability.

▶ What you'll see: the larger logit receives about 96.4% of the routing mass.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a coupling chart.
plt.bar(["upper0", "upper1"], prob_b5, color=["seagreen", "gray"])  # Visualize the softmax split.
plt.ylim(0, 1); plt.title("Basic 5: softmax routing")  # Bound probabilities.
plt.ylabel("c_ij")  # Label coupling values.
plt.show()  # Display the chart.

▶ What you'll see: one route dominates because its logit is much larger.

👀 Takeaway: softmax turns absolute scores into relative routing decisions.

### Basic 6 — Check row-normalized couplings

**Goal.** Verify that couplings sum to 1 across upper capsules for each lower capsule, because routing conserves each part’s evidence budget. We build it in 2 steps.

In [ ]:
B_b6 = np.array([[0.0, 0.0], [0.2, 1.1], [0.6, 0.3]])  # Define routing logits for three lower capsules.
E_b6 = np.exp(B_b6 - np.max(B_b6, axis=1, keepdims=True))  # Stabilize exponentials by subtracting row maxima.
C_b6 = E_b6 / E_b6.sum(axis=1, keepdims=True)  # Softmax over candidate upper capsules.
print("couplings:\n", np.round(C_b6, 3))  # Inspect routing probabilities.

▶ What you'll see: each row is a probability distribution over two upper capsules.

In [ ]:
row_sums_b6 = C_b6.sum(axis=1)  # Sum each lower capsule's outgoing probabilities.
print("row sums:", np.round(row_sums_b6, 3))  # Verify conservation of routing mass.
assert np.allclose(row_sums_b6, np.ones(3))  # Ensure every row sums to 1.
plt.figure(figsize=(4, 3)); plt.imshow(C_b6, cmap="viridis", aspect="auto")  # Visualize the coupling matrix.
plt.colorbar(label="coupling"); plt.title("Basic 6: row-normalized couplings"); plt.show()

▶ What you'll see: the heatmap shows probabilities, and the printed row sums are all 1.

👀 Takeaway: routing chooses among parents separately for each lower capsule.

### Basic 7 — Weight votes by couplings

**Goal.** Compute one upper pre-activation from routed votes, because $s_j=\sum_i c_{ij}\hat u_{j|i}$ is the core routing sum. We build it in 2 steps.

In [ ]:
votes_b7 = np.array([[0.8, 0.08], [0.5, 0.26], [0.39, 0.51]])  # Three votes aimed at the same upper capsule.
c_b7 = np.array([0.5, 0.7, 0.4])  # Coupling weights from three lower capsules to that upper capsule.
weighted_b7 = c_b7[:, None] * votes_b7  # Scale each vote by its route strength.
print("weighted votes:\n", np.round(weighted_b7, 3))  # Inspect individual contributions.

▶ What you'll see: stronger couplings create larger vote contributions.

In [ ]:
s_b7 = weighted_b7.sum(axis=0)  # Sum routed votes into one upper pre-activation vector.
print("s_j:", np.round(s_b7, 3))  # Inspect the collected evidence vector.
assert np.allclose(np.round(s_b7, 3), np.array([0.906, 0.426]))  # Verify the weighted sum.
plt.figure(figsize=(4, 3)); plt.bar(["axis0", "axis1"], s_b7, color="purple")  # Plot pre-activation coordinates.
plt.title("Basic 7: routed sum s_j"); plt.ylabel("coordinate value"); plt.show()

▶ What you'll see: the upper pre-activation is the coordinate-wise sum of weighted votes.

👀 Takeaway: capsule routing is a weighted vote aggregation, not a max over parts.

### Basic 8 — Squash a capsule vector

**Goal.** Apply the capsule squash nonlinearity, because output length should behave like bounded activation while direction remains meaningful. We build it in 3 steps.

In [ ]:
s_b8 = np.array([0.906, 0.426])  # Use a routed pre-activation vector.
norm_b8 = np.linalg.norm(s_b8)  # Compute its input length.
scale_b8 = norm_b8 ** 2 / (1 + norm_b8 ** 2)  # Compute the saturating length factor.
print("input norm:", round(norm_b8, 3), "scale:", round(scale_b8, 3))  # Inspect squash ingredients.

▶ What you'll see: the raw vector is close to length 1, so its squashed length is near 0.5.

In [ ]:
v_b8 = scale_b8 * s_b8 / (norm_b8 + 1e-8)  # Preserve direction and set bounded length.
print("squashed vector:", np.round(v_b8, 3))  # Inspect the output capsule.
print("output norm:", round(float(np.linalg.norm(v_b8)), 3))  # Inspect bounded activation.
assert round(float(np.linalg.norm(v_b8)), 3) == round(scale_b8, 3)  # Verify output length equals scale.

▶ What you'll see: the output points the same way as `s_b8` but has smaller bounded length.

In [ ]:
plt.figure(figsize=(4, 3.4))  # Create a before-after vector plot.
plt.arrow(0, 0, s_b8[0], s_b8[1], head_width=0.035, color="gray", length_includes_head=True, label="s")  # Raw vector.
plt.arrow(0, 0, v_b8[0], v_b8[1], head_width=0.035, color="seagreen", length_includes_head=True, label="v")  # Squashed vector.
plt.xlim(0, 1.0); plt.ylim(0, 0.55); plt.legend(); plt.title("Basic 8: squash keeps direction")
plt.show()

▶ What you'll see: the green vector lies on the same ray but is shorter.

👀 Takeaway: squashing makes capsule activations smooth, bounded, and pose-preserving.

### Basic 9 — Compute agreement with a dot product

**Goal.** Measure whether a vote agrees with an upper capsule output, because dynamic routing reinforces aligned directions. We build it in 2 steps.

In [ ]:
vote_b9 = np.array([0.8, 0.08])  # Define one lower capsule's vote for an upper capsule.
v_upper_b9 = np.array([0.42, 0.19])  # Define the current upper capsule output.
agreement_b9 = float(vote_b9 @ v_upper_b9)  # Compute dot-product agreement.
print("agreement:", round(agreement_b9, 3))  # Inspect alignment strength.
assert round(agreement_b9, 3) == 0.351  # Verify the dot product.

▶ What you'll see: a positive agreement score because the vote and upper output point similarly.

In [ ]:
plt.figure(figsize=(4, 3.4))  # Create an alignment plot.
plt.arrow(0, 0, vote_b9[0], vote_b9[1], head_width=0.035, color="teal", length_includes_head=True, label="vote")  # Vote vector.
plt.arrow(0, 0, v_upper_b9[0], v_upper_b9[1], head_width=0.035, color="orange", length_includes_head=True, label="upper v")  # Upper output.
plt.legend(); plt.title("Basic 9: dot-product agreement"); plt.xlim(0, 0.9); plt.ylim(0, 0.3); plt.show()

▶ What you'll see: the two arrows point in roughly the same direction.

👀 Takeaway: agreement is high when part predictions align with the current whole hypothesis.

### Basic 10 — Estimate capsule memory cost

**Goal.** Convert capsule counts and dimensions into memory, because routing stores many vectors and deep-learning math lands on hardware. We build it in 2 steps.

In [ ]:
n_caps_b10 = 3  # Use the lesson's tiny activation block count.
dim_b10 = 128  # Use the lesson's vector length.
bytes_per_float_b10 = 4  # Use 32-bit floats.
mem_kb_b10 = n_caps_b10 * dim_b10 * bytes_per_float_b10 / 1024  # Convert bytes to KB.
print("memory KB:", round(mem_kb_b10, 3))  # Inspect the activation memory.
assert round(mem_kb_b10, 3) == 1.5  # Verify the lesson memory number.

▶ What you'll see: three 128-D float32 capsule vectors occupy 1.5 KB.

In [ ]:
caps_grid_b10 = np.array([3, 30, 300])  # Compare tiny, medium, and larger capsule counts.
mem_grid_b10 = caps_grid_b10 * dim_b10 * bytes_per_float_b10 / 1024  # Compute memory for each count.
plt.figure(figsize=(4, 3)); plt.bar([str(x) for x in caps_grid_b10], mem_grid_b10, color="crimson")  # Plot growth.
plt.title("Basic 10: capsule activation memory"); plt.xlabel("capsules"); plt.ylabel("KB"); plt.show()

▶ What you'll see: memory grows linearly with capsule count and vector dimension.

👀 Takeaway: capsule vectors carry more structure than scalars, so shape and memory bookkeeping matter.

## 🟡 Easy

### Easy 1 — Run one complete routing pass

**Goal.** Combine votes, softmax couplings, routed sums, and squash once, because one routing pass is the smallest complete capsule computation. We build it in 4 steps.

In [ ]:
u_e1 = np.array([[0.8, 0.1], [0.2, 0.7], [0.6, 0.3]])  # Define three lower capsule outputs.
W_e1 = np.array([[[1.0, 0.0], [0.0, 0.8]], [[0.4, 0.6], [0.7, 0.2]], [[0.2, 0.9], [0.8, 0.1]], [[0.9, 0.1], [0.1, 0.9]], [[0.6, 0.3], [0.2, 0.7]], [[0.3, 0.7], [0.9, 0.1]]]).reshape(3, 2, 2, 2)  # Define transformations.
votes_e1 = np.einsum("ijab,ib->ija", W_e1, u_e1)  # Compute lower-to-upper prediction vectors.
print("votes shape:", votes_e1.shape)  # Inspect the vote tensor shape.

▶ What you'll see: three lower capsules each vote for two upper capsules with two pose coordinates.

In [ ]:
b_e1 = np.zeros((3, 2))  # Start routing logits with no preference.
exp_e1 = np.exp(b_e1)  # Equal logits give equal exponentials.
c_e1 = exp_e1 / exp_e1.sum(axis=1, keepdims=True)  # Softmax couplings.
print("initial couplings:\n", c_e1)  # Inspect the uniform routing distribution.
assert np.allclose(c_e1, 0.5)  # Verify equal routing at initialization.

▶ What you'll see: every lower capsule splits its evidence 50/50.

In [ ]:
s_e1 = np.einsum("ij,ija->ja", c_e1, votes_e1)  # Sum weighted votes for each upper capsule.
norm_e1 = np.linalg.norm(s_e1, axis=1, keepdims=True)  # Compute upper pre-activation lengths.
v_e1 = (norm_e1 ** 2 / (1 + norm_e1 ** 2)) * s_e1 / (norm_e1 + 1e-8)  # Squash upper capsule outputs.
print("upper outputs:\n", np.round(v_e1, 3))  # Inspect the routed outputs.
print("output lengths:", np.round(np.linalg.norm(v_e1, axis=1), 3))  # Inspect class-like activations.

▶ What you'll see: two bounded upper capsule vectors after one pass.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a length comparison chart.
plt.bar(["upper0", "upper1"], np.linalg.norm(v_e1, axis=1), color=["seagreen", "darkorange"])  # Plot capsule activations.
plt.ylim(0, 1); plt.title("Easy 1: one routing pass output lengths"); plt.ylabel("||v_j||"); plt.show()

▶ What you'll see: the taller bar is the upper capsule with stronger routed evidence.

👀 Takeaway: one routing pass maps part capsules into bounded whole-capsule activations.

### Easy 2 — Compare one and three routing iterations

**Goal.** Show how dynamic routing changes couplings over iterations, because agreement should make routes more selective. We build it in 4 steps.

In [ ]:
votes_e2 = np.array([[[0.8, 0.08], [0.38, 0.58]], [[0.5, 0.26], [0.25, 0.65]], [[0.39, 0.51], [0.27, 0.45]]])  # Fixed votes for 3 lower x 2 upper.
b_e2 = np.zeros((3, 2))  # Initialize routing logits uniformly.
history_e2 = []  # Store couplings for each iteration.
print("vote tensor shape:", votes_e2.shape)  # Inspect the tensor dimensions.

▶ What you'll see: the routing problem has three parts and two possible wholes.

In [ ]:
for it_e2 in range(3):  # Perform three dynamic-routing iterations.
    E_e2 = np.exp(b_e2 - np.max(b_e2, axis=1, keepdims=True))  # Stable exponentials for softmax.
    C_e2 = E_e2 / E_e2.sum(axis=1, keepdims=True)  # Couplings for this iteration.
    S_e2 = np.einsum("ij,ija->ja", C_e2, votes_e2)  # Routed upper pre-activations.
    N_e2 = np.linalg.norm(S_e2, axis=1, keepdims=True)  # Lengths before squash.
    V_e2 = (N_e2 ** 2 / (1 + N_e2 ** 2)) * S_e2 / (N_e2 + 1e-8)  # Squashed upper outputs.
    b_e2 = b_e2 + np.einsum("ija,ja->ij", votes_e2, V_e2)  # Agreement update.
    history_e2.append(C_e2.copy())  # Save current couplings.
print("iteration 1 couplings:\n", np.round(history_e2[0], 3))  # Inspect start.
print("iteration 3 couplings:\n", np.round(history_e2[-1], 3))  # Inspect after agreement.

▶ What you'll see: later couplings are no longer exactly uniform.

In [ ]:
change_e2 = history_e2[-1] - history_e2[0]  # Measure coupling movement.
print("coupling change:\n", np.round(change_e2, 3))  # Inspect how routes shifted.
assert np.allclose(history_e2[-1].sum(axis=1), np.ones(3))  # Verify rows still sum to 1.

▶ What you'll see: agreement changes probabilities but preserves each row’s total mass.

In [ ]:
plt.figure(figsize=(4.6, 3))  # Create a routing trajectory plot.
plt.plot([h_e2[0, 0] for h_e2 in history_e2], marker="o", label="lower0→upper0")  # Track one route.
plt.plot([h_e2[1, 1] for h_e2 in history_e2], marker="s", label="lower1→upper1")  # Track another route.
plt.ylim(0, 1); plt.xlabel("iteration"); plt.ylabel("coupling"); plt.title("Easy 2: routing trajectories"); plt.legend(); plt.show()

▶ What you'll see: agreement slowly moves routing probabilities away from the uniform starting point.

👀 Takeaway: dynamic routing is iterative soft assignment driven by vote-output agreement.

### Easy 3 — Compute capsule margin loss

**Goal.** Penalize class capsule lengths with margins, because present classes should be long and absent classes should be short. We build it in 3 steps.

In [ ]:
lengths_e3 = np.array([0.82, 0.18, 0.04])  # Three class capsule lengths.
target_e3 = np.array([1.0, 0.0, 0.0])  # Class 0 is present and the other classes are absent.
m_plus_e3, m_minus_e3, lam_e3 = 0.9, 0.1, 0.5  # Define margin-loss constants.
print("lengths:", lengths_e3)  # Inspect class activations before loss.

▶ What you'll see: the true class length is high but still below the positive margin.

In [ ]:
pos_e3 = target_e3 * np.maximum(0, m_plus_e3 - lengths_e3) ** 2  # Present-class penalty.
neg_e3 = lam_e3 * (1 - target_e3) * np.maximum(0, lengths_e3 - m_minus_e3) ** 2  # Absent-class penalty.
loss_e3 = pos_e3 + neg_e3  # Combine margin terms per class.
print("positive terms:", np.round(pos_e3, 4))  # Inspect present-class penalty.
print("negative terms:", np.round(neg_e3, 4))  # Inspect absent-class penalties.
print("total loss:", round(float(loss_e3.sum()), 4))  # Inspect final loss.
assert round(float(loss_e3.sum()), 4) == 0.0096  # Verify concrete total.

▶ What you'll see: class 0 and class 1 contribute penalties, while the very short absent class does not.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a class-loss breakdown plot.
plt.bar(["class0", "class1", "class2"], loss_e3, color=["seagreen", "orange", "gray"])  # Plot per-class loss.
plt.title("Easy 3: margin loss by class"); plt.ylabel("loss term"); plt.show()

▶ What you'll see: the main penalties come from a true class below 0.9 and an absent class above 0.1.

👀 Takeaway: capsule supervision often acts on vector lengths, not directly on raw logits.

### Easy 4 — Normalize a routing signal

**Goal.** Standardize the lesson signal, because routing and gradients are sensitive to scale. We build it in 3 steps.

In [ ]:
score_e4 = 3.7  # Use the lesson signal after gating.
mean_e4 = 1.0  # Use the lesson normalization mean.
var_e4 = 0.25  # Use the lesson normalization variance.
eps_e4 = 1e-5  # Add epsilon for numerical safety.
print("score, mean, variance:", score_e4, mean_e4, var_e4)  # Inspect normalization inputs.

▶ What you'll see: the raw score is much larger than the chosen mean.

In [ ]:
z_e4 = (score_e4 - mean_e4) / np.sqrt(var_e4 + eps_e4)  # Compute normalized value.
print("normalized score:", round(z_e4, 3))  # Inspect standard-unit scale.
assert round(z_e4, 3) == 5.4  # Verify the lesson number.

▶ What you'll see: the signal is 5.4 normalized units above the mean.

In [ ]:
scores_e4 = np.array([0.4, 1.0, 2.0, 3.7])  # Compare several possible routing signals.
z_grid_e4 = (scores_e4 - mean_e4) / np.sqrt(var_e4 + eps_e4)  # Normalize each signal.
plt.figure(figsize=(4, 3)); plt.bar([str(s) for s in scores_e4], z_grid_e4, color="purple")  # Plot standardized scores.
plt.axhline(0, color="black", linewidth=0.8); plt.title("Easy 4: normalized routing signals"); plt.ylabel("z-score"); plt.show()

▶ What you'll see: values above the mean become positive z-scores, and the 3.7 signal is extreme.

👀 Takeaway: scale control can change the effective strength of routing and gradients even when formulas look correct.

### Easy 5 — Take the lesson gradient step

**Goal.** Apply one scalar gradient-descent update, because capsule routing still depends on repeated small parameter changes. We build it in 3 steps.

In [ ]:
theta_e5 = 2.0  # Start from the lesson scalar parameter.
eta_e5 = 0.09  # Use the lesson learning rate.
grad_e5 = 1.35  # Use the lesson gradient.
step_e5 = eta_e5 * grad_e5  # Compute the amount subtracted.
print("step size:", round(step_e5, 3))  # Inspect the update magnitude.
assert round(step_e5, 3) == 0.121  # Floating-point rounding displays this product as 0.121.

▶ What you'll see: the update is small relative to the parameter value.

In [ ]:
theta_new_e5 = theta_e5 - step_e5  # Move opposite the gradient.
print("updated theta:", round(theta_new_e5, 3))  # Inspect the new parameter.
assert round(theta_new_e5, 3) == 1.879  # Verify the lesson update.

▶ What you'll see: the parameter moves from 2.000 to 1.879.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a before-after parameter plot.
plt.bar(["before", "after"], [theta_e5, theta_new_e5], color=["gray", "seagreen"])  # Visualize the small nudge.
plt.title("Easy 5: one gradient step"); plt.ylabel("parameter value"); plt.show()

▶ What you'll see: the after bar is slightly lower, not dramatically different.

👀 Takeaway: learning changes capsule behavior through many controlled nudges, not one large jump.

## 🔴 Advanced

### Advanced 1 — Route a batch of examples

**Goal.** Run routing for two inputs at once, because real capsule layers process minibatches while keeping vote and coupling shapes straight. We build it in 4 steps.

In [ ]:
votes_a1 = np.array([[[[0.8, 0.1], [0.2, 0.6]], [[0.4, 0.3], [0.5, 0.2]], [[0.3, 0.5], [0.2, 0.4]]],
                     [[[0.2, 0.7], [0.6, 0.1]], [[0.3, 0.6], [0.5, 0.4]], [[0.4, 0.2], [0.7, 0.3]]]])  # batch x lower x upper x dim.
b_a1 = np.zeros(votes_a1.shape[:3])  # Routing logits for each batch item, lower capsule, and upper capsule.
print("votes shape:", votes_a1.shape)  # Inspect batch routing shape.
assert votes_a1.shape == (2, 3, 2, 2)  # Verify dimensions.

▶ What you'll see: two examples, three lower capsules, two upper capsules, and 2-D votes.

In [ ]:
for it_a1 in range(3):  # Run three batched routing iterations.
    E_a1 = np.exp(b_a1 - np.max(b_a1, axis=2, keepdims=True))  # Softmax numerator over upper capsules.
    C_a1 = E_a1 / E_a1.sum(axis=2, keepdims=True)  # Couplings with rows summing to 1 per example.
    S_a1 = np.einsum("bij,bijd->bjd", C_a1, votes_a1)  # Routed sums for each batch and upper capsule.
    N_a1 = np.linalg.norm(S_a1, axis=2, keepdims=True)  # Upper capsule lengths before squash.
    V_a1 = (N_a1 ** 2 / (1 + N_a1 ** 2)) * S_a1 / (N_a1 + 1e-8)  # Squashed outputs.
    b_a1 = b_a1 + np.einsum("bijd,bjd->bij", votes_a1, V_a1)  # Agreement update.
print("output lengths:\n", np.round(np.linalg.norm(V_a1, axis=2), 3))  # Inspect class-like lengths.

▶ What you'll see: each example has two upper capsule activation lengths.

In [ ]:
row_sums_a1 = C_a1.sum(axis=2)  # Sum couplings across upper capsules.
print("coupling row sums:\n", np.round(row_sums_a1, 3))  # Verify probability conservation.
assert np.allclose(row_sums_a1, np.ones((2, 3)))  # Every lower capsule routes all its evidence.

▶ What you'll see: every lower capsule in every batch item has outgoing mass 1.

In [ ]:
plt.figure(figsize=(4.5, 3))  # Create a batch output heatmap.
plt.imshow(np.linalg.norm(V_a1, axis=2), cmap="viridis", aspect="auto")  # Visualize output lengths.
plt.colorbar(label="||v_j||"); plt.xlabel("upper capsule"); plt.ylabel("batch example")
plt.title("Advanced 1: batched capsule lengths"); plt.show()

▶ What you'll see: the heatmap compares the two class-capsule lengths for each example.

👀 Takeaway: routing generalizes cleanly to minibatches if the softmax axis and tensor shapes are explicit.

### Advanced 2 — Sweep routing iterations

**Goal.** Measure how output lengths change with more routing iterations, because too few iterations may underuse agreement and too many add cost. We build it in 4 steps.

In [ ]:
votes_a2 = np.array([[[0.8, 0.08], [0.38, 0.58]], [[0.5, 0.26], [0.25, 0.65]], [[0.39, 0.51], [0.27, 0.45]]])  # Fixed toy votes.
iteration_grid_a2 = np.array([1, 2, 3, 5])  # Test several routing depths.
lengths_by_iter_a2 = []  # Store final upper lengths for each depth.
print("iteration grid:", iteration_grid_a2)  # Inspect the sweep settings.

▶ What you'll see: the experiment compares shallow and deeper routing loops.

In [ ]:
for n_iter_a2 in iteration_grid_a2:  # Loop over routing depths.
    b_a2 = np.zeros((3, 2))  # Reset logits for a fair comparison.
    for it_a2 in range(n_iter_a2):  # Run the requested number of iterations.
        E_a2 = np.exp(b_a2 - np.max(b_a2, axis=1, keepdims=True))  # Stable softmax numerator.
        C_a2 = E_a2 / E_a2.sum(axis=1, keepdims=True)  # Couplings.
        S_a2 = np.einsum("ij,ijd->jd", C_a2, votes_a2)  # Routed sums.
        N_a2 = np.linalg.norm(S_a2, axis=1, keepdims=True)  # Pre-activation lengths.
        V_a2 = (N_a2 ** 2 / (1 + N_a2 ** 2)) * S_a2 / (N_a2 + 1e-8)  # Squash.
        b_a2 = b_a2 + np.einsum("ijd,jd->ij", votes_a2, V_a2)  # Agreement update.
    lengths_by_iter_a2.append(np.linalg.norm(V_a2, axis=1))  # Save final lengths.
print("lengths by iterations:\n", np.round(lengths_by_iter_a2, 3))  # Inspect the sweep result.

▶ What you'll see: output lengths shift as routing has more chances to reinforce agreement.

In [ ]:
lengths_by_iter_a2 = np.array(lengths_by_iter_a2)  # Convert list to array for plotting.
assert lengths_by_iter_a2.shape == (4, 2)  # Verify one pair of upper lengths per iteration setting.
print("shape:", lengths_by_iter_a2.shape)  # Inspect the stored result shape.

▶ What you'll see: the sweep stores four rows and two upper-capsule columns.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a routing-depth plot.
plt.plot(iteration_grid_a2, lengths_by_iter_a2[:, 0], marker="o", label="upper0")  # Track first upper capsule.
plt.plot(iteration_grid_a2, lengths_by_iter_a2[:, 1], marker="s", label="upper1")  # Track second upper capsule.
plt.xlabel("routing iterations"); plt.ylabel("output length"); plt.title("Advanced 2: routing-depth sweep"); plt.legend(); plt.show()

▶ What you'll see: extra iterations change class lengths, but each iteration costs another agreement pass.

👀 Takeaway: routing iterations are a capacity-cost knob that should be chosen deliberately.

### Advanced 3 — Show a missing scale problem

**Goal.** Compare routing with normal and oversized votes, because agreement dot products can become too strong when vector scale is not controlled. We build it in 4 steps.

In [ ]:
votes_base_a3 = np.array([[[0.8, 0.08], [0.38, 0.58]], [[0.5, 0.26], [0.25, 0.65]], [[0.39, 0.51], [0.27, 0.45]]])  # Normal-scale votes.
scales_a3 = np.array([1.0, 5.0])  # Compare ordinary and oversized votes.
final_couplings_a3 = []  # Store final couplings under each scale.
print("scales:", scales_a3)  # Inspect scale settings.

▶ What you'll see: the experiment will isolate vote magnitude as the changed variable.

In [ ]:
for scale_a3 in scales_a3:  # Route with each vote scale.
    votes_a3 = scale_a3 * votes_base_a3  # Scale all votes.
    b_a3 = np.zeros((3, 2))  # Reset logits.
    for it_a3 in range(3):  # Run routing.
        E_a3 = np.exp(b_a3 - np.max(b_a3, axis=1, keepdims=True))  # Stable softmax.
        C_a3 = E_a3 / E_a3.sum(axis=1, keepdims=True)  # Couplings.
        S_a3 = np.einsum("ij,ijd->jd", C_a3, votes_a3)  # Routed sums.
        N_a3 = np.linalg.norm(S_a3, axis=1, keepdims=True)  # Lengths.
        V_a3 = (N_a3 ** 2 / (1 + N_a3 ** 2)) * S_a3 / (N_a3 + 1e-8)  # Squash.
        b_a3 = b_a3 + np.einsum("ijd,jd->ij", votes_a3, V_a3)  # Agreement update.
    final_couplings_a3.append(C_a3.copy())  # Save final couplings.
print("normal final C:\n", np.round(final_couplings_a3[0], 3))  # Inspect normal scale.
print("large final C:\n", np.round(final_couplings_a3[1], 3))  # Inspect large scale.

▶ What you'll see: oversized votes make the softmax routes sharper because agreements are larger.

In [ ]:
sharpness_a3 = np.array([np.mean(np.max(C_a3, axis=1)) for C_a3 in final_couplings_a3])  # Average winning probability.
print("average winning coupling:", np.round(sharpness_a3, 3))  # Inspect route sharpness.
assert sharpness_a3[1] >= sharpness_a3[0]  # Larger scale should not make this demo less sharp.

▶ What you'll see: the large-scale case has a higher average winning coupling.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a sharpness comparison plot.
plt.bar(["scale 1", "scale 5"], sharpness_a3, color=["seagreen", "crimson"])  # Compare route decisiveness.
plt.ylim(0.5, 1); plt.title("Advanced 3: scale sharpens routing"); plt.ylabel("mean max coupling"); plt.show()

▶ What you'll see: the oversized votes produce more decisive routing, which can be unstable if accidental.

👀 Takeaway: controlling vector scale is part of making dynamic routing train reliably.

### Advanced 4 — Train a tiny capsule classifier head

**Goal.** Optimize class-capsule lengths with a finite-difference gradient, because the training objective ultimately changes weights to reduce margin loss. We build it in 5 steps.

In [ ]:
features_a4 = np.array([[1.0, 0.2], [0.2, 1.0]])  # Two toy examples with two input features.
targets_a4 = np.array([[1.0, 0.0], [0.0, 1.0]])  # One-hot class targets for two class capsules.
W_a4 = np.array([[0.7, 0.1], [0.2, 0.6]])  # Tiny linear map from features to class capsule lengths proxy.
print("initial W:\n", W_a4)  # Inspect trainable weights.

▶ What you'll see: a tiny two-class head with four trainable numbers.

In [ ]:
def loss_a4(W_local_a4):  # Define margin loss for this toy head.
    logits_a4 = features_a4 @ W_local_a4  # Compute class scores from features.
    lengths_a4 = 1 / (1 + np.exp(-logits_a4))  # Squash scores to (0,1) length proxies.
    pos_a4 = targets_a4 * np.maximum(0, 0.9 - lengths_a4) ** 2  # Positive class margin penalty.
    neg_a4 = 0.5 * (1 - targets_a4) * np.maximum(0, lengths_a4 - 0.1) ** 2  # Negative class margin penalty.
    return float(np.sum(pos_a4 + neg_a4))  # Total loss over examples and classes.

initial_loss_a4 = loss_a4(W_a4)  # Evaluate starting loss.
print("initial loss:", round(initial_loss_a4, 4))  # Inspect objective before training.

▶ What you'll see: the loss is positive because class lengths are not yet at the desired margins.

In [ ]:
eps_a4 = 1e-4  # Small finite-difference step.
grad_a4 = np.zeros_like(W_a4)  # Prepare gradient matrix.
for r_a4 in range(W_a4.shape[0]):  # Loop over rows.
    for c_a4 in range(W_a4.shape[1]):  # Loop over columns.
        bump_a4 = np.zeros_like(W_a4)  # Create a single-parameter perturbation.
        bump_a4[r_a4, c_a4] = eps_a4  # Bump one weight.
        grad_a4[r_a4, c_a4] = (loss_a4(W_a4 + bump_a4) - loss_a4(W_a4 - bump_a4)) / (2 * eps_a4)  # Central difference.
print("finite-difference gradient:\n", np.round(grad_a4, 4))  # Inspect loss sensitivity.

▶ What you'll see: each weight receives a direction that would increase loss; descent moves opposite it.

In [ ]:
eta_a4 = 0.5  # Choose a visible but stable learning rate for this tiny example.
W_new_a4 = W_a4 - eta_a4 * grad_a4  # Take one gradient-descent step.
new_loss_a4 = loss_a4(W_new_a4)  # Evaluate updated loss.
print("new loss:", round(new_loss_a4, 4))  # Inspect loss after one step.
assert new_loss_a4 < initial_loss_a4  # Verify the update improved the objective.

▶ What you'll see: one finite-difference descent step lowers the capsule-style margin loss.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a loss comparison plot.
plt.bar(["before", "after"], [initial_loss_a4, new_loss_a4], color=["gray", "seagreen"])  # Compare objective values.
plt.title("Advanced 4: training lowers margin loss"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the after bar is lower, confirming that the update moved in a useful direction.

👀 Takeaway: whether gradients are analytic or estimated, the training job is to make correct capsule lengths longer and incorrect ones shorter.

### Advanced 5 — Compare routing cost with a scalar layer

**Goal.** Count multiply-like operations for votes and routing, because capsule expressiveness costs more than scalar activations. We build it in 4 steps.

In [ ]:
n_lower_a5 = 32  # Number of lower capsules.
n_upper_a5 = 10  # Number of upper capsules.
d_in_a5 = 8  # Lower capsule dimension.
d_out_a5 = 16  # Upper capsule vote dimension.
iters_a5 = np.array([1, 2, 3, 5])  # Routing iteration counts to compare.
print("shape:", n_lower_a5, n_upper_a5, d_in_a5, d_out_a5)  # Inspect layer dimensions.

▶ What you'll see: the cost model uses a modest capsule layer shape.

In [ ]:
vote_mults_a5 = n_lower_a5 * n_upper_a5 * d_in_a5 * d_out_a5  # Matrix-vector multiply cost for all votes.
routing_mults_a5 = iters_a5 * n_lower_a5 * n_upper_a5 * d_out_a5  # Approximate agreement and weighted-sum cost per iteration.
total_mults_a5 = vote_mults_a5 + routing_mults_a5  # Total rough cost.
print("vote mults:", vote_mults_a5)  # Inspect transformation cost.
print("total mults by iteration:", total_mults_a5)  # Inspect routing-depth cost.
assert vote_mults_a5 == 40960  # Verify concrete vote cost.

▶ What you'll see: vote transforms dominate the base cost, and routing adds cost per iteration.

In [ ]:
scalar_units_a5 = n_upper_a5 * d_out_a5  # A rough scalar layer with the same output width.
scalar_mults_a5 = n_lower_a5 * d_in_a5 * scalar_units_a5  # Dense scalar-layer cost.
ratio_a5 = total_mults_a5 / scalar_mults_a5  # Compare capsule cost to scalar dense cost.
print("scalar mults:", scalar_mults_a5)  # Inspect baseline cost.
print("capsule/scalar ratios:", np.round(ratio_a5, 2))  # Inspect relative cost.

▶ What you'll see: the capsule layer costs more because every lower capsule votes for every upper capsule.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a cost curve.
plt.plot(iters_a5, ratio_a5, marker="o", color="crimson")  # Plot relative cost by routing iterations.
plt.xlabel("routing iterations"); plt.ylabel("cost ratio vs scalar layer"); plt.title("Advanced 5: routing cost grows with iterations"); plt.show()

▶ What you'll see: extra routing iterations increase cost roughly linearly after votes are formed.

👀 Takeaway: capsule networks trade extra computation and memory for structured vector representations and routing agreement.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Capsules represent parts as vectors and use routing to agree on wholes.

Capsules replace scalar features with vectors, then route lower-level votes to upper-level capsules by agreement. This is a gap topic, so the notebook focuses on the lesson's stated $c_{ij}$ softmax, $s_j$ aggregation, and the pitfall that routing cost is not magic accuracy.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(42)
random.seed(42)

def clf_digits_ladder():
    """A harder image-as-tabular classification ladder for DL topics (part 6).

    D1 XOR -> D2 blobs -> D3 noisy moons -> D4 sklearn digits (10-class, 64-D) ->
    D5 digits with label noise + feature noise (distribution shift).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def split_scale(X, y):
    stratify = y if min(np.bincount(y)) >= 2 else None
    x_tr, x_te, y_tr, y_te = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_softmax_linear(x_tr, y_tr, x_te, epochs=220, lr=0.25, mask=None, theta0=None, quant_bits=None):
    classes = np.unique(y_tr)
    n_classes = int(classes.max()) + 1
    rng = np.random.default_rng(7)
    if theta0 is None:
        W = rng.normal(0.0, 0.05, size=(x_tr.shape[1], n_classes))
        b = np.zeros(n_classes)
    else:
        W = theta0[0].copy()
        b = theta0[1].copy()
    if mask is None:
        mask = np.ones_like(W)
    Y = one_hot(y_tr, n_classes)
    for epoch in range(epochs):
        logits = x_tr @ (W * mask) + b
        probs = softmax(logits)
        grad_logits = (probs - Y) / len(y_tr)
        grad_W = x_tr.T @ grad_logits
        grad_b = grad_logits.sum(axis=0)
        if quant_bits is not None:
            grad_W = quantize_fixed(grad_W, quant_bits)
            grad_b = quantize_fixed(grad_b, quant_bits)
        W = W - lr * grad_W * mask
        b = b - lr * grad_b
    scores = x_te @ (W * mask) + b
    return scores.argmax(axis=1), (W, b)


def quantize_fixed(values, bits):
    values = np.asarray(values, dtype=float)
    levels = 2 ** bits - 1
    clipped = np.clip(values, -2.0, 2.0)
    scaled = np.round((clipped + 2.0) * levels / 4.0)
    return scaled * 4.0 / levels - 2.0


def mlp_predict(x_tr, y_tr, x_te, hidden=(16,), alpha=0.0001, max_iter=260):
    clf = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.02,
        max_iter=max_iter,
        random_state=3,
    )
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def ladder_preview(rungs):
    rows = []
    for name, X, y in rungs:
        rows.append((name, X.shape, int(len(np.unique(y)))))
    for name, shape, classes in rows:
        print(f"{name:34s} shape={shape} classes={classes}")
    print("D1 sample X:")
    print(rungs[0][1])
    print("D1 labels:")
    print(rungs[0][2])


def evaluate_accuracy_ladder(method):
    rows = []
    rungs = clf_digits_ladder()
    for name, X, y in rungs:
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        preds, artifact = method(x_tr, y_tr, x_te, name)
        acc = accuracy_score(y_te, preds)
        rows.append({"name": name, "metric": acc, "artifact": artifact, "X": X, "y": y})
    for row in rows:
        print(f"{row['name']:34s} accuracy={row['metric']:.3f}")
    return rows


def plot_results(rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, row in zip(axes[0], rows):
        X = row["X"]
        y = row["y"]
        if X.shape[1] > 2:
            pts = PCA(n_components=2, random_state=0).fit_transform(X)
        else:
            pts = X
        ax.scatter(pts[:, 0], pts[:, 1], c=y, s=12, cmap="tab10", alpha=0.75)
        ax.set_title(row["name"].split(" (")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    xs = np.arange(1, len(rows) + 1)
    ys = [row["metric"] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel(ylabel)
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## The concept, built once (D1)
$$c_{ij}=\mathrm{softmax}_j(b_{ij}),\qquad s_j=\sum_i c_{ij}\hat u_{j|i}$$

Dynamic routing starts with equal logits $b_{ij}$, computes routing weights, aggregates votes, then increases logits where votes agree with the upper capsule output.

In [ ]:
def squash(vectors):
    norm = np.linalg.norm(vectors, axis=-1, keepdims=True)
    scale = norm ** 2 / (1.0 + norm ** 2)
    return scale * vectors / (norm + 1e-8)


def capsule_routing(votes, iterations=2):
    logits = np.zeros(votes.shape[:2])
    history = []
    for step in range(iterations):
        coeffs = softmax(logits)
        summed = np.einsum("ij,ijd->jd", coeffs, votes)
        outputs = squash(summed)
        agreement = np.einsum("ijd,jd->ij", votes, outputs)
        logits = logits + agreement
        history.append((coeffs.copy(), summed.copy(), outputs.copy()))
    return history[-1][0], history[-1][1], history[-1][2], history


votes_demo = np.array([
    [[1.0, 0.0], [0.2, 0.1]],
    [[0.8, 0.2], [0.1, 0.9]],
])
coeff_demo, summed_demo, out_demo, hist_demo = capsule_routing(votes_demo, iterations=1)
assert np.allclose(coeff_demo, [[0.5, 0.5], [0.5, 0.5]])
assert np.allclose(summed_demo[0], [0.9, 0.1])
assert np.allclose(summed_demo[1], [0.15, 0.5])
print("routing c_ij:")
print(coeff_demo)
print("s_j:")
print(summed_demo)

The assertion above pins the notebook to the lesson's worked numbers before we scale the same idea up the ladder.

In [ ]:
print('D1 concept verified for 6.29')

## The dataset ladder
We use the shared F5 `clf_digits_ladder()` exactly: XOR, blobs, noisy moons, real digits, then noisy shifted digits.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1-D5

In [ ]:
def capsule_predict(x_tr, y_tr, x_te, iterations=2):
    classes = np.unique(y_tr)
    centers = []
    for cls in classes:
        centers.append(x_tr[y_tr == cls].mean(axis=0))
    centers = np.array(centers)
    def score(X):
        sims = X @ centers.T / max(1.0, X.shape[1])
        logits = np.zeros_like(sims)
        for step in range(iterations):
            coeffs = softmax(logits)
            logits = logits + coeffs * sims
        return logits
    return score(x_te).argmax(axis=1), {"centers": centers, "iterations": iterations}


def capsule_method(x_tr, y_tr, x_te, name):
    preds, artifact = capsule_predict(x_tr, y_tr, x_te, iterations=2)
    return preds, artifact


rows = evaluate_accuracy_ladder(capsule_method)

## Results visualization
Top row: rung artifacts in two dimensions. Bottom-left: the one tracked metric from D1 to D5.

In [ ]:
plot_results(rows, 'Capsule routing accuracy across D1-D5')

## Pitfall on the hardest rung
Pitfall on D5: treating routing as magic. More routing iterations add compute, and on a small CPU-safe image ladder they can plateau or even hurt validation accuracy.

In [ ]:
name, X5, y5 = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X5, y5)
for iterations in [1, 2, 3, 4]:
    preds, _ = capsule_predict(x_tr, y_tr, x_te, iterations=iterations)
    acc = accuracy_score(y_te, preds)
    cost = iterations * x_tr.shape[0] * len(np.unique(y_tr))
    print(f"iterations={iterations} accuracy={acc:.3f} relative_cost={cost}")
print("Fix: choose the cheapest iteration count on validation, not the largest count by default.")

## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline such as majority-class accuracy or untrained random predictions.
- Cheap sanity check: rerun with the same seed and confirm the D1 arithmetic assertions still pass.
- Ablation: turn off the key idea (generated context, routing, spikes, validation search, reset, capacity sweep, or loss scaling) and expect the hardest-rung metric to worsen or become less reliable.
- Failure signals: unstable curves, shape mismatches, nearly constant predictions, or a D5 result that improves only by using training labels for selection.

Practice prompts:
1. Change one hyperparameter in the pitfall cell and explain whether the metric moved for the reason the lesson predicts.

In [ ]:
# Try it here.

2. Replace D5 with a smaller subset and predict which failure signal becomes easier or harder to see.

In [ ]:
# Try it here.

3. Add one baseline row to the summary curve and decide whether the specialized method earned its complexity.

In [ ]:
# Try it here.